# Playwright with JavaScript — Day 3
## November 7, 2026 | Auto-waiting and Synchronization

**Practice website:** https://testautomationpractice.blogspot.com/

**Additional page used in this class:** https://testautomationpractice.blogspot.com/p/gui-elements-ajax-hidden.html



### Learning objectives
By the end of this class, students should be able to:
1. Explain why synchronization is required in browser automation.
2. Understand Playwright's auto-waiting and actionability checks.
3. Use web-first assertions to wait for UI states.
4. Use locator waits for visible, hidden, attached and detached states.
5. Understand navigation, load states and network waits.
6. Handle elements that are initially hidden and become visible after a click.
7. Diagnose timeout errors without adding unnecessary fixed delays.

> **Website accuracy note:** The examples use the page labels and controls currently published on the practice site. Public websites can change. Run the locator-count/visibility checks before relying on a selector, and re-check the live DOM if the page has changed.

## 2. Open the actual practice website

The main practice page is:

https://testautomationpractice.blogspot.com/

For this lesson, use the page linked in the footer as **Hidden Elements & AJAX**:

https://testautomationpractice.blogspot.com/p/gui-elements-ajax-hidden.html

The page contains these named controls:
- Input Box 1
- Input Box 2
- Checkbox 1 and Checkbox 2
- Toggle Input Box 2
- Toggle Checkbox 2
- Load AJAX Content
- Status: Ready

The main page also contains a **Dynamic Web Table** and a **Dynamic Button** section.

### First verification test


```javascript
const { test, expect } = require('@playwright/test');

test('Open the synchronization practice page', async ({ page }) => {
    await page.goto(
        'https://testautomationpractice.blogspot.com/p/gui-elements-ajax-hidden.html'
    );

    await expect(page).toHaveTitle(/Automation Testing Practice/i);
    await expect(
        page.getByRole('button', { name: 'Toggle Input Box 2' })
    ).toBeVisible();

    await expect(
        page.getByRole('button', { name: 'Load AJAX Content' })
    ).toBeVisible();
});
```

**Teaching point:** `page.goto()` waits for the navigation load condition by default. The assertions then verify the specific controls needed by the test.

## 3. Why synchronization is needed

A web application does not finish every operation at the same time.

Examples from this practice page:
- Input Box 2 starts hidden and is revealed after clicking a toggle button.
- Checkbox 2 can be revealed or hidden.
- The Load AJAX Content button initiates a dynamic content operation.
- The Dynamic Web Table displays values that can change.

**Synchronization** means coordinating the test with the application state.

A **race condition** occurs when the test tries to continue before the application has reached the state the test expects.

A **flaky test** passes and fails inconsistently because of timing or other unstable conditions.

### A timing mistake

```javascript
await page.getByRole('button', {
    name: 'Load AJAX Content'
}).click();

console.log(await page.locator('#some-result').textContent());
```

The code assumes the result is already available immediately after the click. That assumption is unsafe unless the test waits for the actual result.

## 4. Playwright auto-waiting

Before supported locator actions, Playwright automatically waits for the target to become actionable.

For `locator.click()`, Playwright checks that:
- The locator resolves to exactly one element.
- The element is visible.
- The element is stable (not moving or animating).
- The element receives pointer events (not covered by another element).
- The element is enabled.
- The element is not inert.

### Example from the practice website

```javascript
await page.goto(
    'https://testautomationpractice.blogspot.com/p/gui-elements-ajax-hidden.html'
);

const toggleButton = page.getByRole('button', {
    name: 'Toggle Input Box 2'
});

await toggleButton.click();
```

No fixed delay is required before this click. Playwright waits for the button's actionability.

### Important distinction
Auto-waiting before a click does **not** mean that a separate AJAX operation started by the click has finished. After clicking, wait for the expected result or state.

## 5. Actionability checks vary by action

```javascript
// Click: waits for the button to be actionable.
await page.getByRole('button', {
    name: 'Toggle Checkbox 2'
}).click();

// Fill: waits for an input to be visible, enabled and editable.
await page.locator('input[type="text"]').first().fill('Playwright');

// Check: waits for the checkbox to be actionable.
await page.locator('input[type="checkbox"]').first().check();
```

Different actions have different actionability requirements. Do not assume `click()`, `fill()` and `check()` all use exactly the same checklist.

### When an action times out, investigate:
1. Is the locator correct?
2. Does it match exactly one element?
3. Is the element visible?
4. Is it enabled and editable where required?
5. Is an overlay covering it?
6. Is an animation or page transition in progress?

Avoid immediately using `force: true`. It bypasses checks and can hide a genuine user-facing problem.

## 6. Fixed wait vs condition-based wait

### Fixed wait

```javascript
await page.waitForTimeout(3000);
```

This always pauses for three seconds.

- If the content is ready in 300 ms, two and a half seconds are wasted.
- If the content takes four seconds, the test still continues too early.
- It can make the test slow and flaky.

Use fixed waits temporarily while debugging, not as the normal solution.

### Condition-based wait

```javascript
await page.locator('.result-message').waitFor({
    state: 'visible'
});
```

Locator wait states:
- `attached` — element exists in the DOM.
- `detached` — element is removed from the DOM.
- `visible` — element is visible.
- `hidden` — element is hidden or detached.

### Preferred for test verification

```javascript
await expect(page.locator('.result-message')).toBeVisible();
```

A web-first assertion retries until it passes or reaches its assertion timeout.

## 7. Practical 1 — Reveal and hide Input Box 2

On the Hidden Elements & AJAX page, Input Box 2 is initially hidden. The page provides a button named **Toggle Input Box 2**.

First, inspect the input elements in the relevant part of the page. The following locator uses the observed order of the page's text inputs: Input Box 1, then Input Box 2. Verify the count and order in the browser before running the exercise.

```javascript
const textInputs = page.locator('input[type="text"]');

console.log('Text input count:', await textInputs.count());

// On this page, the second text input is Input Box 2.
// Verify this mapping in DevTools if the website changes.
const inputBox2 = textInputs.nth(1);

console.log('Initially visible:', await inputBox2.isVisible());

await page.getByRole('button', {
    name: 'Toggle Input Box 2'
}).click();

await expect(inputBox2).toBeVisible();

await page.getByRole('button', {
    name: 'Toggle Input Box 2'
}).click();

await expect(inputBox2).toBeHidden();
```

### What students should understand
- `isVisible()` returns an immediate Boolean result.
- `toBeVisible()` retries until the element becomes visible or times out.
- `toBeHidden()` retries until the element is hidden or detached.
- The click action itself auto-waits for the toggle button to be actionable.

**Selector verification:** If the second text input is not Input Box 2 in the live DOM, inspect the input labels and scope the locator to the correct page section rather than guessing a new index.

## 8. Practical 2 — Reveal and hide Checkbox 2

The same page has **Toggle Checkbox 2**.

The first two checkbox inputs on this page correspond to Checkbox 1 and Checkbox 2 in the sample UI section. Verify the count/order in DevTools before running; the page may be updated over time.

```javascript
const checkboxes = page.locator('input[type="checkbox"]');

console.log('Checkbox count:', await checkboxes.count());

const checkbox2 = checkboxes.nth(1);

console.log('Initially visible:', await checkbox2.isVisible());

await page.getByRole('button', {
    name: 'Toggle Checkbox 2'
}).click();

await expect(checkbox2).toBeVisible();

await checkbox2.check();
await expect(checkbox2).toBeChecked();

await page.getByRole('button', {
    name: 'Toggle Checkbox 2'
}).click();

await expect(checkbox2).toBeHidden();
```

### Discussion
- Why does `check()` not need a fixed wait before it?
- What is the difference between hidden and unchecked?
- Why should we verify that the locator refers to Checkbox 2 before interacting with it?

## 9. Practical 3 — Load AJAX Content: how to synchronize correctly

The page has a button named **Load AJAX Content** and initially displays **Status: Ready**.

The important learning point is this: clicking the button only starts the operation. The test must wait for an observable signal that represents completion.

### Step 1 — inspect the live behavior
1. Open the Hidden Elements & AJAX page manually.
2. Note the exact status text before clicking.
3. Click **Load AJAX Content**.
4. Observe the status changes and the content that appears.
5. Inspect the newly added content in DevTools.
6. Identify a stable locator for the final content—not a temporary loading message.

### Step 2 — use the observed final state
Use this pattern after you have identified the exact final content locator from the live page:

```javascript
await page.getByRole('button', {
    name: 'Load AJAX Content'
}).click();

// Replace this example locator with the exact stable locator
// you identified for the content after it has loaded.
const loadedContent = page.locator('YOUR_VERIFIED_RESULT_SELECTOR');

await expect(loadedContent).toBeVisible();
```

**Do not copy `YOUR_VERIFIED_RESULT_SELECTOR` literally.** It is a teaching placeholder, not a selector from the website.

### Why this is deliberate
The page's public text confirms the button and initial status, but the final AJAX result selector/text must be verified against the live DOM. This avoids teaching students an invented locator that returns zero matches.

## 10. Web-first assertions: the most useful synchronization tool

Import `expect` from `@playwright/test`.

```javascript
const { test, expect } = require('@playwright/test');
```

Common web-first assertions:

```javascript
await expect(locator).toBeVisible();
await expect(locator).toBeHidden();
await expect(locator).toBeEnabled();
await expect(locator).toBeDisabled();
await expect(locator).toHaveText('Expected text');
await expect(locator).toContainText('Part of expected text');
await expect(locator).toHaveValue('Expected value');
await expect(locator).toBeChecked();
await expect(locator).toHaveCount(4);
```

These assertions retry until the expected condition is met or the assertion timeout is reached.

### Immediate check vs retrying assertion

```javascript
// Immediate check. Returns true or false now.
const visibleNow = await locator.isVisible();
console.log(visibleNow);

// Retrying assertion. Waits for visibility.
await expect(locator).toBeVisible();
```

Use `isVisible()` when you specifically need an immediate Boolean decision. Use `toBeVisible()` when the test must wait for and verify visibility.

## 11. Practical 4 — Dynamic Web Table on the main page

Open the main practice page:

https://testautomationpractice.blogspot.com/

The page has a **Static Web Table** followed by a **Dynamic Web Table**. The dynamic table displays changing values, so avoid asserting a particular CPU, memory or network number.

### Inspect tables first

```javascript
const tables = page.locator('table');

console.log('Number of tables:', await tables.count());

for (let i = 0; i < await tables.count(); i++) {
    console.log(
        'Table', i,
        await tables.nth(i).innerText()
    );
}
```

This is an inspection exercise. Identify which table is the Dynamic Web Table from its heading and the live DOM before choosing its locator.

### After verifying the dynamic table locator

```javascript
const dynamicTable = page.locator('YOUR_VERIFIED_DYNAMIC_TABLE_SELECTOR');
const rows = dynamicTable.locator('tbody tr');

// Wait for at least one data row to be available.
// If the table is expected to contain four rows, assert four instead.
await expect(rows).not.toHaveCount(0);

console.log('Current data rows:', await rows.count());
console.log(await dynamicTable.innerText());
```

**Do not copy the placeholder selector literally.** Use DevTools to identify the actual table element. Dynamic values may change between test runs, so assert stable structure or meaningful labels rather than exact live metrics.

## 12. Page load states

```javascript
await page.goto('https://testautomationpractice.blogspot.com/', {
    waitUntil: 'domcontentloaded'
});

await page.waitForLoadState('load');
```

Common load states:
- `domcontentloaded` — the HTML document has been parsed and the DOMContentLoaded event fired.
- `load` — the page load event fired.
- `networkidle` — network connections have been idle for a period.

### Important
`page.goto()` waits for the `load` state by default. A page can still update after that because JavaScript may fetch data later.

`networkidle` is not a universal signal that an application is ready. Background requests can continue, and network quiet does not prove that the content your test needs is visible.

Prefer a specific assertion:

```javascript
await page.goto('https://testautomationpractice.blogspot.com/');
await expect(
    page.getByRole('button', { name: 'START' })
).toBeVisible();
```

The `START` button is listed in the page's Dynamic Button section. This example verifies the button is available; it does not assume what happens after clicking it.

## 13. Waiting for URL changes, popups and network responses

These are general synchronization patterns. Use them when the behavior under test actually triggers the corresponding event.

### URL change

```javascript
await Promise.all([
    page.waitForURL('**/expected-path'),
    page.getByRole('link', { name: 'Open page' }).click()
]);
```

### Popup

```javascript
const popupPromise = page.waitForEvent('popup');

await page.getByRole('button', { name: 'Open popup' }).click();

const popup = await popupPromise;
await popup.waitForLoadState('domcontentloaded');
```

### Network response

```javascript
const responsePromise = page.waitForResponse(response =>
    response.url().includes('/verified-api-endpoint') &&
    response.status() === 200
);

await page.getByRole('button', { name: 'Load data' }).click();

const response = await responsePromise;
console.log(response.status());
```

Register the event wait **before** the click so a fast event is not missed.

The URL fragment `/verified-api-endpoint` is a placeholder. Inspect the Network tab and use the actual request URL only when teaching a response-wait exercise on a page whose endpoint has been verified.

## 14. Timeout handling

Timeouts are maximum waiting limits. They are not fixed delays.

### Locator action timeout

```javascript
await page.getByRole('button', {
    name: 'Load AJAX Content'
}).click({ timeout: 10000 });
```

### Assertion timeout

```javascript
await expect(page.locator('YOUR_VERIFIED_RESULT_SELECTOR'))
    .toBeVisible({ timeout: 10000 });
```

### Test timeout

```javascript
test.setTimeout(30000);
```

### Configuration example

```javascript
const { defineConfig } = require('@playwright/test');

module.exports = defineConfig({
    timeout: 30000,
    expect: {
        timeout: 5000
    },
    use: {
        actionTimeout: 10000,
        navigationTimeout: 15000
    }
});
```

Increasing a timeout can help when a known operation is genuinely slow. It does not fix an incorrect selector, missing element or broken application.

## 15. Complete runnable practice test — toggle controls

This test focuses on the controls whose labels and toggle buttons are present on the Hidden Elements & AJAX page. It checks the input/checkbox ordering before using the second matching element.

```javascript
const { test, expect } = require('@playwright/test');

test('Synchronize with hidden input and checkbox controls', async ({ page }) => {
    await page.goto(
        'https://testautomationpractice.blogspot.com/p/gui-elements-ajax-hidden.html'
    );

    const inputToggle = page.getByRole('button', {
        name: 'Toggle Input Box 2'
    });

    const checkboxToggle = page.getByRole('button', {
        name: 'Toggle Checkbox 2'
    });

    await expect(inputToggle).toBeVisible();
    await expect(checkboxToggle).toBeVisible();

    const textInputs = page.locator('input[type="text"]');
    const checkboxes = page.locator('input[type="checkbox"]');

    // Verify these indexes against the live DOM if the page changes.
    await expect(textInputs).toHaveCount(2);
    await expect(checkboxes).toHaveCount(2);

    const inputBox2 = textInputs.nth(1);
    const checkbox2 = checkboxes.nth(1);

    await expect(inputBox2).toBeHidden();
    await expect(checkbox2).toBeHidden();

    await inputToggle.click();
    await expect(inputBox2).toBeVisible();

    await checkboxToggle.click();
    await expect(checkbox2).toBeVisible();

    await checkbox2.check();
    await expect(checkbox2).toBeChecked();

    await inputToggle.click();
    await expect(inputBox2).toBeHidden();

    await checkboxToggle.click();
    await expect(checkbox2).toBeHidden();
});
```

### If the count assertion fails
Do not remove the assertion blindly. Inspect the page's inputs and scope the locator to the sample UI section. The purpose of the count assertion is to catch selector drift before the test interacts with the wrong control.

## 16. Common mistakes and better alternatives

| Avoid | Prefer | Why |
|---|---|---|
| `waitForTimeout(5000)` before every click | Let locator actions auto-wait | Avoids unnecessary fixed delays |
| Read dynamic text immediately | `expect(locator).toHaveText(...)` | Retries until expected text appears |
| Assume `goto()` means AJAX is finished | Assert the actual loaded content | Navigation and later data loading are different |
| Use `networkidle` for every page | Wait for the specific UI state | Background requests may continue |
| Use a guessed selector | Inspect DOM and verify locator count | Prevents zero matches or wrong element |
| Increase timeout for every failure | Diagnose locator and application state | A long timeout does not fix a bad selector |
| Use `isVisible()` when you need to wait | `toBeVisible()` | `isVisible()` is immediate |

### Debugging commands

```javascript
console.log('Input count:', await page.locator('input').count());
console.log('Button count:', await page.getByRole('button').count());

console.log(
    await page.getByRole('button', {
        name: 'Load AJAX Content'
    }).isVisible()
);
```

Use these to inspect what Playwright can currently find. Keep temporary inspection logs out of the final clean test unless they provide useful diagnostics.

## 17. Student assignment — 30 minutes

### Task A — Verify the page
1. Navigate to the Hidden Elements & AJAX page.
2. Verify the page title.
3. Verify that the Toggle Input Box 2 and Toggle Checkbox 2 buttons are visible.

### Task B — Hidden input
1. Inspect the text input count and order.
2. Identify Input Box 2.
3. Verify that it is initially hidden.
4. Click Toggle Input Box 2.
5. Verify that it becomes visible.
6. Click the toggle again and verify it becomes hidden.

### Task C — Hidden checkbox
1. Inspect the checkbox count and order.
2. Identify Checkbox 2.
3. Reveal it using Toggle Checkbox 2.
4. Check it and verify it is checked.
5. Hide it again and verify the hidden state.

### Task D — AJAX investigation
1. Click Load AJAX Content manually.
2. Record the initial status, intermediate status (if any), and final content.
3. Inspect the DOM and identify a stable locator for the final content.
4. Write a Playwright assertion that waits for that content.
5. Do not use a fixed sleep as the final solution.

### Submission
- One `.spec.js` file.
- Screenshot or terminal output of the test run.
- The verified locator used for the AJAX result.
- A short explanation of why auto-waiting does not automatically wait for all AJAX work.

## 18. Quick knowledge check

1. What is synchronization in browser automation?
2. What does auto-waiting do before a click?
3. Name four actionability checks for `click()`.
4. Why is `waitForTimeout(5000)` usually a poor solution?
5. What is the difference between hidden and detached?
6. What is the difference between `isVisible()` and `toBeVisible()`?
7. Does `page.goto()` guarantee that later AJAX content is ready?
8. Why should we inspect the actual result element after clicking Load AJAX Content?
9. Why can `networkidle` be unreliable on modern sites?
10. Does increasing a timeout fix a wrong locator?

### Answer key
1. Coordinating test steps with the application's state.
2. Waits for the target element to become actionable.
3. Visible, stable, receives events, enabled (and the locator must resolve uniquely).
4. It always waits the same duration, regardless of readiness.
5. Hidden means present but not visible, or possibly detached; detached specifically means removed from the DOM. For a precise distinction, use `toBeAttached()` / `not.toBeAttached()` alongside visibility checks.
6. `isVisible()` returns an immediate Boolean; `toBeVisible()` retries.
7. No. Navigation lifecycle completion is different from later asynchronous application work.
8. To use a real, stable locator rather than inventing one.
9. Background requests can keep the network active or make network quiet unrelated to UI readiness.
10. No.

## 19. Day 3 recap and official references

### Recap
- Browser applications update asynchronously.
- Playwright auto-waits for actionability before supported locator actions.
- Web-first assertions retry and verify the expected UI state.
- Use locator waits when you need a specific attached/visible/hidden/detached state.
- A click can start AJAX work; it does not prove the operation has completed.
- Verify dynamic page locators against the live DOM.
- Avoid fixed sleeps in normal tests.
- A timeout is a maximum wait limit, not a guaranteed delay.

### Practice pages
- Main page: https://testautomationpractice.blogspot.com/
- Hidden Elements & AJAX: https://testautomationpractice.blogspot.com/p/gui-elements-ajax-hidden/

### Official Playwright documentation
- Auto-waiting and actionability: https://playwright.dev/docs/actionability
- Assertions: https://playwright.dev/docs/test-assertions
- Locator API: https://playwright.dev/docs/api/class-locator
- Page API: https://playwright.dev/docs/api/class-page
- Navigation: https://playwright.dev/docs/navigations

### Next class — Day 4
**Dropdowns, tables and web controls**